# Concepts

This page introduces the core ideas behind SysSimX. It provides a high-level mental model and explains the key concepts needed to build and simulate heterogeneous systems.


## What is SysSimX?

SysSimX is a Python framework for heterogeneous co-simulation. It lets you connect models from different tools (FMUs, FEM solvers, OpenSim, or custom Python components) into a single system and advance them consistently in time.


## Mental Model: Component, Connection, System, Step

- **Component**: A model with inputs, outputs, parameters, and (optionally) internal state.
- **Connection**: A directed signal path from an output port to an input port.
- **System**: A container that manages components, connections, and execution order.
- **Step**: A single time advance of the system with a fixed `dt`.


## Architectural Overview

![SysSimX Architecture](../_static/SysSimX_Simplified.svg)

## Component Lifecycle

Every component follows the same lifecycle:

1. **Initialize**: `initialize(t0)` sets the component state.
2. **Advance**: `do_step(t, dt)` advances the component state.
3. **Publish**: outputs are updated after each step.
4. **Record**: outputs are stored in history for plotting and analysis.

Inputs are set via `set_inputs(...)` before each step. You can retrieve data with `get_outputs()` or `get_history()`.


## Ports and Units

Ports define the interface of a component. Each port has a **direction** (in/out) and may have a **unit**.

Connections handle unit conversion automatically using Pint. This is important when combining models that use different unit conventions.


## Execution Order and Direct Feedthrough

SysSimX computes an execution order based on dependencies between components.

- **Direct feedthrough** means an output depends on inputs at the same time step.
- Components with direct feedthrough must run after the components that provide their inputs.

The system builds a dependency graph and derives a parallelizable execution order.


## Algebraic Loops

If components with direct feedthrough are connected in a cycle, an **algebraic loop** is formed. SysSimX detects these loops and solves them iteratively using the IJCSA method.


## Master Algorithms

SysSimX supports multiple stepping strategies:

- **Jacobi**: parallel updates, simple but may lag direct feedthrough signals.
- **Gauss-Seidel**: sequential updates, uses latest values within a step.
- **Hybrid**: event-driven stepping with zero-crossing detection.

The system selects the appropriate algorithm based on component capabilities and event sources.


## Component Types
SysSimX provides component wrappers for common external tools:

- **FMUComponent** for FMI 2.0 Co-Simulation models
- **FEMComponent** for NGSolve-based finite element models
- **OpenSimComponent** for musculoskeletal biomechanics models
- **Custom Python components** for quick prototyping


## Events and Hybrid Simulation

Some systems have discrete events (contacts, switches, impacts). SysSimX supports zero-crossing detection and event localization to handle these cases accurately.
